In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.transforms as mtransforms
import matplotlib.colors as mcolors
from openpmd_viewer import OpenPMDTimeSeries
from scipy.ndimage import gaussian_filter
import scipy.constants as sc

# Interpolate B onto particle positions to get more accurate perp vs par components
from scipy.interpolate import RegularGridInterpolator

In [ ]:
def add_external_parameter_box(fig=None, params=None, theme='neutral', ax=None, dx=0, dy=0):
    """
    Places a text box outside the plot and dynamically forces the 
    graph to shrink just enough to prevent clipping or overlap.
    """
    if fig is None:
        fig = plt.gcf()
    if params is None:
        params = {"Status": "Default"}
        
    # 1. Themes definition
    themes = {
        'neutral':  {'bg': '#f8f9fa', 'border': '#cccccc', 'text': '#333333'},
        'success':  {'bg': '#e6f4ea', 'border': '#137333', 'text': '#137333'},
        'warning':  {'bg': '#fef7e0', 'border': '#b06000', 'text': '#b06000'},
        'critical': {'bg': '#fce8e6', 'border': '#c5221f', 'text': '#c5221f'}
    }
    style = themes.get(theme, themes['neutral'])
    text_str = []
    for key, val in params.items():
        if val > 1000:
            text_str.append(f"{key}: {val:.2e}")
        else:
            text_str.append(f"{key}: {val}")
    text_str = "\n".join(text_str)
    
    # Place text anchored to the right side of the plot box
    if fig:
        t = fig.text(
            0.1 + dx, 0.5 + dy, text_str, fontsize=10, color=style['text'], weight='medium',
            verticalalignment='center', horizontalalignment='left', transform=fig.axes[-1].transAxes,
            bbox=dict(boxstyle="round,pad=0.6", facecolor=style['bg'], edgecolor=style['border'], linewidth=1.5)
        )
    elif ax:
        t = ax.text(
                    0.1 + dx, 0.5 + dy, text_str, fontsize=10, color=style['text'], weight='medium',
                    verticalalignment='center', horizontalalignment='left', transform=ax.transAxes,
                    bbox=dict(boxstyle="round,pad=0.6", facecolor=style['bg'], edgecolor=style['border'], linewidth=1.5)
                )

In [ ]:
from coil_3d import SingleCoil3DConfig
cfg = SingleCoil3DConfig()
static_params = {
    "$n_{stream}$": cfg.n_stream,
    "$v_{drift} (m/s)$": cfg.v_drift,
    "$T_{i}(eV)$": cfg.T_i_eV,
    "$T_{e}(eV)$": cfg.T_e_eV,
    "I(A)": cfg.I,
    "R(m)": cfg.R_coil,
}

In [ ]:
MU0  = sc.mu_0
gamma = 5/3
Ti_eV = cfg.T_i_eV
Te_eV = cfg.T_e_eV
Ti_J = Ti_eV * sc.eV
Te_J = Te_eV * sc.eV
n_stream = cfg.n_stream
v_drift = cfg.v_drift
rho = n_stream * sc.m_p
P_ram  = rho * v_drift**2
P_th = n_stream * (Ti_J + Te_J)
P_tot = P_ram + P_th
I = cfg.I
dia = cfg.dia
R_coil = dia / 2
r_CF = np.sqrt((MU0 * I * R_coil**2 / (2 * np.sqrt(2 * MU0 * P_tot)))**(2/3) - R_coil**2)
print(r_CF)
save_path = './diags'
series_f   = OpenPMDTimeSeries(save_path + '/field_diag')
series_p   = OpenPMDTimeSeries(save_path + '/part_diag')
iterations = series_f.iterations[1:]



# Velocity moments

In [ ]:
def bin_moments(x, z, vx, vy, vz, w, xs, zs):
    dx = xs[1] - xs[0]
    dz = zs[1] - zs[0]
    x_edges = np.r_[xs - 0.5*dx, xs[-1] + 0.5*dx]
    z_edges = np.r_[zs - 0.5*dz, zs[-1] + 0.5*dz]

    counts, _, _ = np.histogram2d(x, z, bins=[x_edges, z_edges], weights=w)

    safe = np.where(counts > 0, counts, 1.0)

    def wmean(v):
        h, _, _ = np.histogram2d(x, z, bins=[x_edges, z_edges], weights=w*v)
        return h / safe

    vx_m, vy_m, vz_m = wmean(vx), wmean(vy), wmean(vz)

    # per-cell variance requires per-particle subtraction of the cell mean
    # need to map each particle to its cell mean first
    ix = np.clip(np.searchsorted(x_edges, x) - 1, 0, len(xs)-1)
    iz = np.clip(np.searchsorted(z_edges, z) - 1, 0, len(zs)-1)

    dvx = vx - vx_m[ix, iz]
    dvy = vy - vy_m[ix, iz]
    dvz = vz - vz_m[ix, iz]

    sigma2_raw, _, _ = np.histogram2d(x, z, bins=[x_edges, z_edges],
                                    weights=w*(dvx**2 + dvy**2 + dvz**2))
    sigma2 = np.where(counts > 0, sigma2_raw / safe, 0.0)

    return sigma2, [vx_m, vy_m, vz_m]

# optimized velocity moments

In [ ]:
def bin_moments(x, z, vx, vy, vz, w, xs, zs):
    dx = xs[1] - xs[0]
    dz = zs[1] - zs[0]
    x_edges = np.r_[xs - 0.5*dx, xs[-1] + 0.5*dx]
    z_edges = np.r_[zs - 0.5*dz, zs[-1] + 0.5*dz]
    # one-time cell index per particle, flattened for bincount
    ix = np.clip(np.searchsorted(x_edges, x) - 1, 0, len(xs)-1)
    iz = np.clip(np.searchsorted(z_edges, z) - 1, 0, len(zs)-1)
    flat = ix * len(zs) + iz
    nbins = len(xs) * len(zs)

    def cell_sum(vals):
        return np.bincount(flat, weights=w*vals, minlength=nbins).reshape(len(xs), len(zs))

    counts = cell_sum(np.ones_like(w))
    safe = np.where(counts > 0, counts, 1.0)

    # first and second moments in the same pass, no per-particle re-subtraction
    sum1 = {c: cell_sum(v) for c, v in zip('xyz', (vx, vy, vz))}
    sum2 = {c: cell_sum(v**2) for c, v in zip('xyz', (vx, vy, vz))}

    means = {c: sum1[c]/safe for c in 'xyz'}
    sigma2 = sum(sum2[c]/safe - means[c]**2 for c in 'xyz')
    sigma2 = np.where(counts > 0, sigma2, 0.0)

    vm = [means[c] for c in 'xyz']

    return sigma2, vm

# Count and v_mean moments, warpx provides T (from third moment) diagnostics

WarpX calculates T using the sigma2 $<v_m>^2 - <v_m^2>$

In [ ]:
def bin_moments(x, z, vx, vy, vz, w, xs, zs):
    dx = xs[1] - xs[0]
    dz = zs[1] - zs[0]
    x_edges = np.r_[xs - 0.5*dx, xs[-1] + 0.5*dx]
    z_edges = np.r_[zs - 0.5*dz, zs[-1] + 0.5*dz]
    # one-time cell index per particle, flattened for bincount
    ix = np.clip(np.searchsorted(x_edges, x) - 1, 0, len(xs)-1)
    iz = np.clip(np.searchsorted(z_edges, z) - 1, 0, len(zs)-1)
    flat = ix * len(zs) + iz
    nbins = len(xs) * len(zs)

    def cell_sum(vals):
        return np.bincount(flat, weights=w*vals, minlength=nbins).reshape(len(xs), len(zs))

    counts = cell_sum(np.ones_like(w))
    safe = np.where(counts > 0, counts, 1.0)

    # first and second moments in the same pass, no per-particle re-subtraction
    sum1 = {c: cell_sum(v) for c, v in zip('xyz', (vx, vy, vz))}
    #sum2 = {c: cell_sum(v**2) for c, v in zip('xyz', (vx, vy, vz))}

    means = {c: sum1[c]/safe for c in 'xyz'}
    # sigma2 = sum(sum2[c]/safe - means[c]**2 for c in 'xyz')
    # sigma2 = np.where(counts > 0, sigma2, 0.0)

    vm = [means[c] for c in 'xyz']

    return vm

# Beta map over time

In [ ]:
# ── Precompute ───────────────────────────────────────────────────
beta_frames = []
xs = zs = None

sigma2s, vms = [], []

for k, it in enumerate(iterations):
    print(f"  Loading {k+1}/{len(iterations)}", end='\r')

    Bx, info = series_f.get_field('B', coord='x', iteration=it, slice_across='y')
    By, _    = series_f.get_field('B', coord='y', iteration=it, slice_across='y')
    Bz, _    = series_f.get_field('B', coord='z', iteration=it, slice_across='y')
    rho, _   = series_f.get_field('rho_stream_i', iteration=it, slice_across='y')
    #rho_bg, _ = series_f.get_field('rho_background_i', iteration=it, slice_across='y')

    x, z, ux, uy, uz, w = series_p.get_particle(var_list=['x', 'z', 'ux', 'uy', 'uz', 'w'], iteration=it)

    xs, zs = info.x, info.z

    B2      = np.maximum(Bx**2 + By**2 + Bz**2, 1e-12)
    n_stream_t       = np.abs(rho + 1e-12) / sc.e
    #n_bg_t = np.abs(rho_bg + 1e-12) / sc.e
    # ux = gamma * v / c
    # https://warpx.readthedocs.io/en/26.01/usage/parameters.html#particle-push-charge-and-current-deposition-field-gathering
    # under: <diag_name>.particle_fields.<field_name>(x,y,z,ux,uy,uz)
    vx, vy, vz = ux * sc.c, uy * sc.c, uz * sc.c
    sigma2, vm = bin_moments(x, z, vx, vy, vz, w, xs, zs)
    sigma2s.append(sigma2)
    vms.append(vm)
    p_i = (1/3) * sc.m_p * (n_stream_t) *  sigma2
    p_e = n_stream * Te_J * (n_stream_t / n_stream)**(gamma)
    #p_bg = n_bg_t * (Ti_J) + n_bg_t * sc.m_p * cfg.v_drift**2
    v_bulk2 = vm[0]**2 + vm[1]**2 + vm[2]**2
    p_ram = n_stream_t * sc.m_p * v_bulk2
    p_tot = (p_i + p_e + p_ram)
    # P_tot = P_ram + P_th = n_stream(v_drift**2 + Ti_J + Te_J)
    beta_th = (p_tot) / (B2 / (2 * MU0))

    beta_frames.append(gaussian_filter(np.log10(np.maximum(beta_th, 1e-3)), sigma=1))
    #beta_frames.append(np.log10(np.maximum(beta_th, 1e-12)))

print("\nDone.")

# ── Animate ──────────────────────────────────────────────────────
fig, (ax, ax_text) = plt.subplots(1, 2, figsize=(10, 5), layout='constrained', width_ratios=(2, 1))
ax_text.axis('off')
im = ax.imshow(beta_frames[0], origin='lower',
               extent=[xs[0], xs[-1], zs[0], zs[-1]],
               vmin=-3, vmax=3, cmap='RdBu_r', aspect='equal')
contour_handle = [ax.contour(xs, zs, beta_frames[0], levels=[0.0],
                             colors='k', linewidths=1.0)]
plt.colorbar(im, ax=ax, label='log₁₀ β_th')
ax.set_xlabel('x (m)')
ax.set_ylabel('z (m)')
title = ax.set_title('HELLO IM" TITLE')

add_external_parameter_box(fig, static_params)

plt.show()

def update(k):
    it = iterations[k]
    im.set_data(beta_frames[k])
    contour_handle[0].remove()
    contour_handle[0] = ax.contour(xs, zs, beta_frames[k], levels=[0.0],
                                   colors='k', linewidths=1.0)
    t_us = series_f.t[k] * 1e6
    title.set_text(f'β_th  step {it}  t = {t_us:.2f} µs')
    return [im]

ani = animation.FuncAnimation(fig, update, frames=len(iterations), interval=100)
ani.save(f'{save_path}/beta_timelapse.mp4', writer='ffmpeg', fps=10, dpi=300)
plt.close()
print(f"Saved: {save_path}/beta_timelapse.mp4")

# Alternate beta map using T_species diagnostic

In [ ]:
# ── Precompute ───────────────────────────────────────────────────
beta_frames = []
xs = zs = None

sigma2s, vms = [], []

for k, it in enumerate(iterations):
    print(f"  Loading {k+1}/{len(iterations)}", end='\r')

    Bx, info = series_f.get_field('B', coord='x', iteration=it, slice_across='y')
    By, _    = series_f.get_field('B', coord='y', iteration=it, slice_across='y')
    Bz, _    = series_f.get_field('B', coord='z', iteration=it, slice_across='y')
    rho, _   = series_f.get_field('rho_stream_i', iteration=it, slice_across='y')
    T_ion, _ = series_f.get_field('T_stream_i', iteration=it, slice_across='y')
    #rho_bg, _ = series_f.get_field('rho_background_i', iteration=it, slice_across='y')

    x, z, ux, uy, uz, w = series_p.get_particle(var_list=['x', 'z', 'ux', 'uy', 'uz', 'w'], iteration=it)
    # ux = gamma * v / c
    # https://warpx.readthedocs.io/en/26.01/usage/parameters.html#particle-push-charge-and-current-deposition-field-gathering
    # under: <diag_name>.particle_fields.<field_name>(x,y,z,ux,uy,uz)
    vx, vy, vz = ux * sc.c, uy * sc.c, uz * sc.c
    vm = bin_moments(x, z, vx, vy, vz, w, info.x, info.z)
    # sigma2s.append(sigma2)
    vms.append(vm)

    xs, zs = info.x, info.z

    B2      = np.maximum(Bx**2 + By**2 + Bz**2, 1e-12)
    n_stream_t       = np.abs(rho + 1e-12) / sc.e
    p_i = n_stream_t * T_ion * sc.eV
    p_e = n_stream * Te_J * (n_stream_t / n_stream)**(gamma)
    #p_bg = n_bg_t * (Ti_J) + n_bg_t * sc.m_p * cfg.v_drift**2
    v_bulk2 = vm[0]**2 + vm[1]**2 + vm[2]**2
    p_ram = n_stream_t * sc.m_p * v_bulk2
    p_tot = (p_i + p_e + p_ram)
    beta_th = (p_tot) / (B2 / (2 * MU0))

    beta_frames.append(gaussian_filter(np.log10(np.maximum(beta_th, 1e-3)), sigma=1))
    #beta_frames.append(np.log10(np.maximum(beta_th, 1e-12)))

print("\nDone.")

# ── Animate ──────────────────────────────────────────────────────
fig, (ax, ax_text) = plt.subplots(1, 2, figsize=(10, 5), layout='constrained', width_ratios=(2, 1))
ax_text.axis('off')
im = ax.imshow(beta_frames[0], origin='lower',
               extent=[xs[0], xs[-1], zs[0], zs[-1]],
               vmin=-3, vmax=3, cmap='RdBu_r', aspect='equal')
contour_handle = [ax.contour(xs, zs, beta_frames[0], levels=[0.0],
                             colors='k', linewidths=1.0)]
plt.colorbar(im, ax=ax, label='log₁₀ β_th')
ax.set_xlabel('x (m)')
ax.set_ylabel('z (m)')
title = ax.set_title('HELLO IM" TITLE')

add_external_parameter_box(fig, static_params)

plt.show()

def update(k):
    it = iterations[k]
    im.set_data(beta_frames[k])
    contour_handle[0].remove()
    contour_handle[0] = ax.contour(xs, zs, beta_frames[k], levels=[0.0],
                                   colors='k', linewidths=1.0)
    t_us = series_f.t[k] * 1e6
    title.set_text(f'β_th  step {it}  t = {t_us:.2f} µs')
    return [im]

ani = animation.FuncAnimation(fig, update, frames=len(iterations), interval=100)
ani.save(f'{save_path}/beta_with_Tdiag_timelapse.mp4', writer='ffmpeg', fps=10, dpi=300)
plt.close()
print(f"Saved: {save_path}/beta_with_Tdiag_timelapse.mp4")

# Temperature diags

## Temp XZ

In [ ]:
# ── Precompute ───────────────────────────────────────────────────
t_frames = []
rho_frames = []
xs = zs = None

for k, it in enumerate(iterations):
    print(f"  Loading {k+1}/{len(iterations)}", end='\r')
    T_ion, info = series_f.get_field('T_stream_i', iteration=it, slice_across='y')
    rho, info = series_f.get_field('rho_stream_i', iteration=it, slice_across='y')

    xs, zs = info.x, info.z

    t_frames.append(gaussian_filter(T_ion, sigma=1))
    rho_frames.append(gaussian_filter((rho + 1e-12) / sc.e, sigma=1))

print("\nDone.")

vmin = np.min([k for k in t_frames])
vmax = np.max([k for k in t_frames])

# ── Animate ──────────────────────────────────────────────────────
fig, (ax_T, ax_rho, ax_text) = plt.subplots(1, 3, figsize=(10, 5), layout='constrained', width_ratios=(2, 2, 1), sharey=True)
ax_text.axis('off')
im_T = ax_T.imshow(t_frames[0], origin='lower',
               extent=[xs[0], xs[-1], zs[0], zs[-1]], vmin=vmin, vmax=vmax,
               cmap='hot', aspect='equal')
plt.colorbar(im_T, ax=ax_T, label='$T_i$')
ax_T.set_xlabel('x (m)')
ax_T.set_ylabel('z (m)')
title_T = ax_T.set_title('HELLO IM" TITLE')

vmin = np.min([k for k in rho_frames])
vmax = np.max([k for k in rho_frames])

im_rho = ax_rho.imshow(rho_frames[0], origin='lower',
                    extent=[xs[0], xs[-1], zs[0], zs[-1]], vmin=vmin, vmax=vmax,
                    cmap='plasma', aspect='equal')

plt.colorbar(im_rho, ax=ax_rho, label=r'$n$')
ax_rho.set_xlabel('x (m)')
#ax_rho.set_ylabel('z (m)')
title_rho = ax_rho.set_title('HELLO IM" TITLE')

add_external_parameter_box(fig=fig, params=static_params, dx=3.5, dy=0.25)

plt.show()

ims = [im_T, im_rho]

def update(k):
    it = iterations[k]
    ims[0].set_data(t_frames[k])
    ims[1].set_data(rho_frames[k])
    t_us = series_f.t[k] * 1e6
    title_T.set_text(f'$T_i$ eV  step {it}  t = {t_us:.2f} µs')
    title_rho.set_text(r'$n$ ' + f'step {it}  t = {t_us:.2f} µs')

ani = animation.FuncAnimation(fig, update, frames=len(iterations), interval=100)
ani.save(f'{save_path}/temp_density_timelapse.mp4', writer='ffmpeg', fps=10, dpi=300)
plt.close()
print(f"Saved: {save_path}/temp_density_timelapse.mp4")

# Diagnosing T diagnostics

In [ ]:
it = iterations[50]
Bx, info = series_f.get_field('B', coord='x', iteration=it)
By, _    = series_f.get_field('B', coord='y', iteration=it)
Bz, _    = series_f.get_field('B', coord='z', iteration=it)
x, y, z, ux, uy, uz, w = series_p.get_particle(var_list=['x', 'y', 'z', 'ux', 'uy', 'uz', 'w'], iteration=it)
print(x.shape)
print(Bx.shape)
xs = info.x
ys = info.y
zs = info.z
P = np.stack([x, y, z], axis=0)
B = np.stack([Bx, By, Bz], axis=0)
V = np.stack([ux, uy, uz], axis=0)
print(P.shape)
print(B.shape) # B.shape = (3, 64, 64)
print(V.shape) # V.shape = (3, 14684142)

# one interpolator per B component over the full 3D grid
# bounds_error=False + NaN fill so out-of-domain particles are visible, not silently wrong
interp_kwargs = dict(bounds_error=False, fill_value=np.nan)
interp_Bx = RegularGridInterpolator((xs, ys, zs), Bx, **interp_kwargs)
interp_By = RegularGridInterpolator((xs, ys, zs), By, **interp_kwargs)
interp_Bz = RegularGridInterpolator((xs, ys, zs), Bz, **interp_kwargs)

# Ensuring correct shape for grid interpolation
print(Bx.shape)                       # e.g. (64, 64, 64) — but which axis is which?
print(len(xs), len(ys), len(zs))      # compare against Bx.shape dims in order

# Determining what is actually possible versus what we find from our data
print("x range:", x.min(), x.max(), "grid x range:", xs.min(), xs.max())
print("y range:", y.min(), y.max(), "grid y range:", ys.min(), ys.max())
print("z range:", z.min(), z.max(), "grid z range:", zs.min(), zs.max())

# pts = P.T  # (N, 3), interpolator wants points as rows

# B_p = np.stack([interp_Bx(pts), interp_By(pts), interp_Bz(pts)], axis=0)  # (3, N)

# # Determining what is actually outside of our interpolation grid
# nan_mask = np.isnan(B_p).any(axis=0)
# print(f"{nan_mask.sum()} / {len(x)} particles ({100*nan_mask.mean():.3f}%) out of bounds")

# # look at their positions specifically
# print(P[:, nan_mask].T[:10])   # first 10 offending particles' (x,y,z)

# Check if we even need these particles
# distance from center along the max-magnitude axis, for the NaN population
# edge_dist = np.max(np.abs(P[:, nan_mask]), axis=0)   # how close to the L=1.5 boundary
# print(edge_dist.min(), edge_dist.max())
# compare to the flux-counter / scraping-diagnostic population if you want to check overlap

for b in ['xhi', 'xlo', 'yhi', 'ylo', 'zhi', 'zlo']:

    left = OpenPMDTimeSeries(save_path + '/boundary_scraping/particles_at_' + b)
    left_it = left.iterations[-1]
    x, y, z, ux, uy, uz, w = left.get_particle(var_list=['x', 'y', 'z', 'ux', 'uy', 'uz', 'w'], iteration=left_it)
    P = np.stack([x, y, z], axis=0)
    edge_dist = np.max(np.abs(P), axis=0)   # how close to the L=1.5 boundary
    print(f"************{b}************")
    print(edge_dist.min(), edge_dist.max())
# compare to the flux-counter / scraping-diagnostic population if you want to check overlap

# is this fraction stable over iterations, or growing (e.g. runaway loss / bad injection balance)?
near_b = []
for it in iterations:
    x_, y_, z_, w_ = series_p.get_particle(var_list=['x','y','z','w'], iteration=it)
    frac_out = np.mean((np.abs(x_) > xs.max()) | (np.abs(y_) > ys.max()) | (np.abs(z_) > zs.max()))
    near_b.append(frac_out)

plt.plot(near_b)
plt.show()

In [ ]:
# arrays to accumulate over the iteration loop — same length/order as your T_perp/T_par time series
frac_out_trace   = []
T_par_trace      = []   # domain-averaged or region-averaged scalar, whatever you're already computing
T_perp_trace     = []
time_trace       = []
T_diff_trace     = []
coherence_min_trace = []
coherence_p5_trace  = []   # 5th percentile — more robust than a single min, less swayed by one weird cell

for it in iterations[:14]:
    print(f'Iterations: {it}/{iterations[-1]}', end='\r')
    Bx, info = series_f.get_field('B', coord='x', iteration=it)
    By, _    = series_f.get_field('B', coord='y', iteration=it)
    Bz, _    = series_f.get_field('B', coord='z', iteration=it)
    x_, y_, z_, ux_, uy_, uz_, w_ = series_p.get_particle(
        var_list=['x','y','z','ux','uy','uz','w'], iteration=it)

    xs, ys, zs = info.x, info.y, info.z

    T, _ = series_f.get_field('T_stream_i', iteration=it)  # or however it's named in your run

    ux_, uy_, uz_ = ux_ * sc.c, uy_ * sc.c, uz_ * sc.c

    P_ = np.stack([x_, y_, z_], axis=0)    # (3, N)
    V_ = np.stack([ux_, uy_, uz_], axis=0) # (3, N)

    # Some lie outside of the possible interpolation, this is more extreme at early states
    # Particles are injected at boundaries, but fields are calculated at cell centers
    # Inherent mismatch of particle position and field "position"
    nan_mask_ = (np.abs(x_) > xs.max()) | (np.abs(y_) > ys.max()) | (np.abs(z_) > zs.max())
    frac_out_trace.append(np.average(nan_mask_, weights=w_))  # WEIGHTED fraction, not raw count —
                                                                 # macroparticle weights may not be uniform,
                                                                 # and weighted is what actually matters physically

    # drop out-of-bounds particles BEFORE any physics calc, not just for the tracking metric
    keep = ~nan_mask_
    P_, V_, w_ = P_[:, keep], V_[:, keep], w_[keep]

    # one interpolator per B component over the full 3D grid
    # bounds_error=False + NaN fill so out-of-domain particles are visible, not silently wrong
    interp_kwargs = dict(bounds_error=False, fill_value=np.nan)
    interp_Bx = RegularGridInterpolator((xs, ys, zs), Bx, **interp_kwargs, method='cubic')
    interp_By = RegularGridInterpolator((xs, ys, zs), By, **interp_kwargs, method='cubic')
    interp_Bz = RegularGridInterpolator((xs, ys, zs), Bz, **interp_kwargs, method='cubic')
    
    pts = P_.T  # (N, 3), interpolator wants points as rows
    B_p = np.stack([interp_Bx(pts), interp_By(pts), interp_Bz(pts)], axis=0)  # (3, N)
    B_mag = np.linalg.norm(B_p, axis=0, keepdims=True)
    b_hat = B_p / B_mag  # (3, N) unit vectors

    # per-particle projection onto local B direction
    # b_hat: (3,) unit vector of local B at particle position (need B field, not just B_ext)
    v_par  = np.sum(V_ * b_hat, axis=0)                       # scalar, signed, sum_rows((3, N) * (3, N)) -> (N, )
    # [None, :] turns it from (N, ) to (1, N), allows for (3, N) broadcasting with b_hat
    # Broadcasting goes right to left, so we ensure right index is equal, so broadcasting
    # occurs at the first/left index
    v_vec_perp = V_ - v_par[None, :] * b_hat         # 2D vector in plane perp to B, 3D without this orientation

    # ============================================================
    # STEP 1: figure out which grid cell each particle sits in
    # ============================================================
    # Per cell weighted bulk means 
    # cell spacing in each direction (grid is uniform, so any adjacent pair works)
    dx, dy, dz = xs[1] - xs[0], ys[1] - ys[0], zs[1] - zs[0]
    # Get indices of each particles position
    # convert each particle's PHYSICAL position into an INTEGER cell index.
    # (position - grid_origin) / cell_spacing = "how many cells in am I", then truncate to int.
    # clip(..., 0, len(xs)-1) guards against particles exactly on/past the last grid point
    # (e.g. floating point edge cases) landing on an out-of-range index.
    ix = np.clip(((P_[0] - xs[0]) / dx).astype(int), 0, len(xs) - 1)
    iy = np.clip(((P_[1] - ys[0]) / dy).astype(int), 0, len(ys) - 1)
    iz = np.clip(((P_[2] - zs[0]) / dz).astype(int), 0, len(zs) - 1)

    # each particle now has an (ix, iy, iz) triplet identifying its cell — a "3D address."
    # ravel_multi_index flattens that 3D address into a SINGLE integer 0..ncells-1,
    # the same way a 3D array is laid out contiguously in memory.
    # This is just a relabeling — cell (ix,iy,iz) and cell_id are the same cell, different notation.
    # Each particle is given a cell number from 0 to ncells - 1
    cell_id = np.ravel_multi_index((ix, iy, iz), (len(xs), len(ys), len(zs))) # shape (N,) — one label per particle

    # total number of distinct cells in the grid — this is the number of "bins" we'll ever have,
    # fixed by the grid, independent of how many particles happen to land in each one
    ncells = len(xs) * len(ys) * len(zs)

    # ============================================================
    # STEP 2: SCATTER — collapse N particles down to ncells bulk values
    #         ("particle → grid": many particles per cell -> one number per cell)
    # ============================================================
    def cell_weighted_mean(q, weights, cell_id, ncells):
        """
        q: (N, )
        weights: (N, )
        cell_id: (N, ) - each particles unique grid id from 0 to ncells - 1

        Computes a weighted average of quantity `q`, grouped by which cell
        each particle belongs to (per cell_id).

        Analogous to WarpX's own two-pass NGP deposition:
        pass 1: sum up (weight * quantity) per cell  -> "num"
        pass 2: sum up weight per cell                -> "den"  (this is just N_array)
        result: num / den = weighted mean per cell

        bincount(cell_id, weights=X) sums X separately for every distinct
        value of cell_id — i.e. "add up X for all particles that share this cell_id."
        This is the vectorized equivalent of looping over cells and summing particles inside.

        minlength=ncells is a FLOOR on the output array length, not a cap:
        bincount by default only makes bins up to the LARGEST cell_id actually seen
        in this iteration's particle data. If the highest-index cell (e.g. a domain
        corner) happens to be empty at this iteration, bincount would return an array
        SHORTER than ncells, which would break the later .reshape(nx,ny,nz).
        minlength=ncells guarantees the output is always exactly length ncells,
        zero-padding any missing/empty cells at the end.
        (cell_id itself can never exceed ncells-1, since ix/iy/iz are already
        clipped into range — so this never creates MORE than ncells bins either.)
        """
        num = np.bincount(cell_id, weights=weights*q, minlength=ncells)   # Σ w_i * q_i, per cell
        den = np.bincount(cell_id, weights=weights, minlength=ncells)     # Σ w_i,       per cell (= N_array)

        # divide, but avoid 0/0 for empty cells: leave those cells at 0 instead of NaN
        return np.divide(num, den, out=np.zeros_like(num), where=den > 0)

    # get bulk/mean parallel and perpendicular velocities
    # apply the scatter operation: for each of the ncells cells, get the local
    # weighted-mean v_par and v_perp (3-component vector) of the particles inside it.
    # Output shape: (ncells,) for v_par_bulk_grid, (3, ncells) for v_perp_bulk_grid.
    v_par_bulk_grid = cell_weighted_mean(v_par, w_, cell_id, ncells)
    v_perp_bulk_grid = np.stack([
        cell_weighted_mean(v_vec_perp[i], w_, cell_id, ncells) for i in range(3)
    ]) # (3, ncells)

    # ============================================================
    # STEP 3: GATHER — broadcast each cell's bulk value back out to
    #         every particle that lives in that cell
    #         ("grid → particle": one number per cell -> back to N particles)
    # ============================================================
    # Gathering the per cell bulk velocities
    # cell_id[i] tells us which cell particle i is in; using it as a fancy index
    # into the (ncells,)-length grid array looks up that cell's bulk value for particle i.
    # Result has shape (N,) again — same length as the original particle arrays —
    # but every particle sharing a cell now carries the SAME bulk value.
    v_par_bulk = v_par_bulk_grid[cell_id]
    # same idea, but gathering all 3 components of the vector bulk perp velocity
    v_perp_bulk_vec = v_perp_bulk_grid[:, cell_id]

    # v_par and v_par_bulk are both scalars per particle (signed, along b_hat).
    # Fluctuation = this particle's v_par minus ITS CELL's local bulk v_par.
    v_par_fluct = v_par - v_par_bulk            # (N,) — signed deviation from local mean
    # Square it: this is the per-particle contribution to the parallel variance.
    # (Squaring a scalar difference is unambiguous — no vector subtlety here,
    #  this is exactly the same operation WarpX does per-component before summing.)
    # this is |δu_par|^2
    v_par_fluct_sq = v_par_fluct**2             # (N,) — always >= 0

    # --- PERPENDICULAR component (vector, 2 DOF worth) ---

    # v_vec_perp and v_perp_bulk_vec are both (3,N) VECTORS (perp-plane component
    # of velocity, still expressed in 3D Cartesian — projecting OUT the parallel
    # part, not literally reducing to 2 numbers).
    # Subtract componentwise (x,y,z each independently) — this is vector subtraction,
    # done BEFORE any squaring, so directional info is preserved through this step.
    v_perp_fluct_vec = v_vec_perp - v_perp_bulk_vec    # (3, N) — deviation VECTOR, still 3 components

    # NOW square each component and sum over the 3 (x,y,z) axes (axis=0).
    # This collapses the vector down to a scalar: |v_perp_fluct_vec|^2 per particle.
    # Squaring+summing AFTER subtracting is what correctly gives |a-b|^2, including cross-term
    # this is |δu_perp|^2
    v_perp_fluct_sq = np.sum(v_perp_fluct_vec**2, axis=0)   # (N,) — scalar per particle, always >= 0

    # At this point:
    # Both v_par_fluct_sq and v_perp_fluct_sq end up as the same kind of quantity: 
    #   a nonnegative scalar per particle, ready to be weighted-averaged into a temperature.

    # 3/2kT = 1/2(mv**2), but 1/2(kT) for par and (kT) for perp
    # T_par: 1 DOF -> no factor of 3
    T_par  = sc.m_p * np.average(v_par_fluct_sq, weights=w_) / sc.e

    # T_perp: 2 DOF -> factor of 2, not 3
    T_perp = sc.m_p * np.average(v_perp_fluct_sq, weights=w_) / (2 * sc.e)
    # append scalar T_par, T_perp for this iteration to their traces
    T_par_trace.append(T_par)
    T_perp_trace.append(T_perp)

    time_trace.append(it)

    T_recombined = (T_par + 2*T_perp) / 3

    v_bulk_grid = np.stack([
        cell_weighted_mean(V_[i], w_, cell_id, ncells) for i in range(3)
    ]) # (3, ncells)
    v_bulk = v_bulk_grid[:, cell_id] # (3, N)
    v_fluct_sq = np.sum((V_ - v_bulk)**2, axis=0) # (N, )
    T_isotropic_warpx = sc.m_p * np.average(v_fluct_sq, weights=w_) / (3 * sc.e)   # 3 DOF -> divide by 3, matches WarpX exactly

    T_diff = abs(T_recombined - T_isotropic_warpx)/T_isotropic_warpx

    print(f"(T_par + 2*T_perp)/3 = {T_recombined:.3f} eV")
    print(f"T_isotropic          = {T_isotropic_warpx:.3f} eV")   # or T_isotropic_warpx
    print(f"relative diff        = {T_diff:.4%}")

    T_diff_trace.append(abs(T_recombined - T_isotropic_warpx)/T_isotropic_warpx)

    B_threshold = 1e-4   # tune based on the min|B| trend you already have; start here, adjust after seeing result

    B_mag = np.reshape(B_mag, -1)
    keep_B = B_mag > B_threshold   # (N,) — same length as your post-NaN-exclusion particle arrays

    # recompute T_par, T_perp using ONLY the high-confidence-|B| particles
    T_par_masked  = sc.m_p * np.average(v_par_fluct_sq[keep_B],  weights=w_[keep_B]) / sc.e
    T_perp_masked = sc.m_p * np.average(v_perp_fluct_sq[keep_B], weights=w_[keep_B]) / (2 * sc.e)
    T_recombined_masked = (T_par_masked + 2*T_perp_masked) / 3

    # compare against the SAME masked population's isotropic T (apples to apples — mask both sides)
    v_fluct_sq_masked = v_fluct_sq[keep_B]   # from your existing isotropic calc
    T_isotropic_masked = sc.m_p * np.average(v_fluct_sq_masked, weights=w_[keep_B]) / (3 * sc.e)

    T_diff_masked = abs(T_recombined_masked - T_isotropic_masked) / T_isotropic_masked
    print(f"excluded fraction: {1 - keep_B.mean():.3%}")
    print(f"T_diff (all)   : {T_diff:.3%}")
    print(f"T_diff (masked): {T_diff_masked:.3%}")

    # Average of b_hat vectors
    # If they are well aligned, |b_hat| -> 1, else -> 0
    b_hat_mean_grid = np.stack([
        cell_weighted_mean(b_hat[i], np.ones_like(w_), cell_id, ncells) for i in range(3)
    ])  # (3, ncells) — NOTE: unweighted-by-norm average of unit vectors, not renormalized

    coherence_grid = np.linalg.norm(b_hat_mean_grid, axis=0)  # (ncells,) -> 1 = fully aligned, 0 = fully scrambled

    # map back to per-particle for inspection, or just look at the distribution/min over cells
    print(f"min coherence: {coherence_grid[coherence_grid>0].min():.3f}")
    print(f"mean coherence: {coherence_grid[coherence_grid>0].mean():.3f}")

    # only look at cells that actually have particles — empty cells give coherence=0 trivially
    # (0 vector average of nothing), which would swamp the real signal
    occupied = cell_weighted_mean(np.ones_like(w_), w_, cell_id, ncells) > 0  # crude occupancy check via den>0 logic
    # simpler: reuse den from cell_weighted_mean directly if you want to avoid recomputing —
    # for now this works, just wasteful; fine at this stage

    coherence_min_trace.append(coherence_grid[occupied].min())
    coherence_p5_trace.append(np.percentile(coherence_grid[occupied], 5))

frac_out_trace = np.array(frac_out_trace)
T_par_trace    = np.array(T_par_trace)
T_perp_trace   = np.array(T_perp_trace)

# Plotting results from above

In [ ]:
fig, ax = plt.subplots()
sc_plot = ax.scatter(coherence_p5_trace, T_diff_trace, c=time_trace, cmap='viridis')
ax.set_xlabel('5th percentile b_hat coherence (per iteration)')
ax.set_ylabel('T_diff (relative error)')
plt.colorbar(sc_plot, label='iteration')

np.corrcoef(coherence_p5_trace, T_diff_trace)[0,1]

# B Lineout over time

In [ ]:
# ── Precompute ───────────────────────────────────────────────────
B_lineouts = []
xs_line    = None

for k, it in enumerate(iterations):
    print(f"  Loading {k+1}/{len(iterations)}", end='\r')

    Bx, info = series_f.get_field('B', coord='x', iteration=it, slice_across=['y', 'z'])
    By, _    = series_f.get_field('B', coord='y', iteration=it, slice_across=['y', 'z'])
    Bz, _    = series_f.get_field('B', coord='z', iteration=it, slice_across=['y', 'z'])

    if xs_line is None:
        xs_line = info.x

    B_mag = np.sqrt(Bx**2 + By**2 + Bz**2)
    B_lineouts.append(gaussian_filter(B_mag, sigma=1))

print("\nDone.")

# ── Animate ──────────────────────────────────────────────────────
B_max = max(b.max() for b in B_lineouts)
print(B_max)

fig, (ax, ax_text) = plt.subplots(1, 2, figsize=(8, 4), width_ratios=(3, 1), layout='constrained')
ax_text.axis('off')
line,  = ax.plot(xs_line, B_lineouts[0], color='steelblue', lw=2)
ax.axhline(0, color='k', lw=0.5, ls='--')
ax.set_ylim(0, B_max * 1.1)
ax.set_xlabel('x (m)')
ax.set_ylabel('|B| (T)')
title = ax.set_title('')

add_external_parameter_box(fig, static_params)

plt.show()

def update(k):
    line.set_ydata(B_lineouts[k])
    t_us = series_f.t[k] * 1e6
    title.set_text(f'|B| lineout  step {iterations[k]}  t = {t_us:.2f} µs')
    return [line]

ani = animation.FuncAnimation(fig, update, frames=len(iterations), interval=100)
ani.save(f'{save_path}/B_lineout_timelapse.mp4', writer='ffmpeg', fps=10, dpi=150)
plt.close()
print(f"Saved: {save_path}/B_lineout_timelapse.mp4")



# Streamplots

In [ ]:
# Precompute Streamplots
B_streamplots = []
xs_line = None
xz_line = None
for k, it in enumerate(iterations):
    print(f"  Loading {k+1}/{len(iterations)}", end='\r')
    Bx, info = series_f.get_field('B', coord='x', iteration=it, slice_across=['y'])
    By, _    = series_f.get_field('B', coord='y', iteration=it, slice_across=['y'])
    Bz, _    = series_f.get_field('B', coord='z', iteration=it, slice_across=['y'])

    xs_line = info.x
    xz_line = info.z 

    B_mag = np.sqrt(Bx**2 + By**2 + Bz**2)
    B_streamplots.append([Bx, Bz, B_mag])

print("\nDone")

# Animate
B_max = max(b[2].max() for b in B_streamplots)

fig, (ax, ax_text) = plt.subplots(1, 2, figsize=(10, 5), layout='constrained', width_ratios=[2, 1])
ax_text.axis('off')
ax.set_aspect('equal')
vmax = np.max([b[2] for b in B_streamplots])
vmin = np.min([b[2] for b in B_streamplots])
norm = mcolors.Normalize(vmin=vmin, vmax=vmax)  # Explicitly enforces 0 as white across all frames
im = ax.streamplot(
    x=info.x,
    y=info.z,
    u=B_streamplots[0][0],
    v=B_streamplots[0][1],
    color=B_streamplots[0][2], norm=norm,
    cmap="viridis", density=2.0, linewidth=1,
    broken_streamlines=True,
)

cbar = fig.colorbar(im.lines, ax=ax, label=r'$|B|$')

ax.set_xlabel('x (m)')
ax.set_ylabel('z (m)')
title = ax.set_title('Iteration 0')

add_external_parameter_box(fig, static_params)

plt.savefig('test.png')
plt.show()

def update(k):
    it = iterations[k]
    ax.cla()
    im = ax.streamplot(
        x=info.x,
        y=info.z,
        u=B_streamplots[k][0],
        v=B_streamplots[k][1],
        color=B_streamplots[k][2],
        cmap="viridis", density=2.0, linewidth=1
    )
    t_us = series_f.t[k] * 1e6
    ax.set_title(f'|B|  step {it}  t = {t_us:.2f} µs')
    ax.set_xlabel('x (m)')
    ax.set_ylabel('z (m)')
    return [im]

ani = animation.FuncAnimation(fig, update, frames=len(iterations), interval=100)
ani.save(f'{save_path}/B_stream_timelapse.mp4', writer='ffmpeg', fps=10, dpi=300)
plt.close()
print(f"Saved: {save_path}/B_stream_timelapse.mp4")

# Face Cusp Losses over time (make a copy if running mid run)

In [ ]:
data   = np.load(f'diags/cusp_flux.npz', allow_pickle=True)
times  = data['times'] * 1e6          # convert to µs
flux_minus = data['flux_minus']               # shape (n_steps, 6)
flux_plus = data['flux_plus']
#labels = data['face_labels']

print(flux_plus.shape)
print(flux_minus.shape)

flux_minus = flux_minus.T

# Total flux across various radii
fig, (ax, ax_text) = plt.subplots(1, 2, figsize=(8, 4), width_ratios=(3, 1))
ax_text.axis('off')

rs = np.linspace(cfg.R_coil / flux_minus.shape[0], cfg.R_coil, flux_plus.shape[1])

print(rs.shape)

lines = []
for k, r in enumerate(rs):
    if k > 0:
        line, = ax.plot(times, flux_minus[k] - flux_minus[k-1], lw = 1.5, label=f'leaving, {0.1*(k):.2f} <= r < {r:.2f}')
    else:
        line, = ax.plot(times, flux_minus[k], lw = 1.5, label=f'leaving, r < {r:.2f}')
    lines.append(line)

line, = ax.plot(times, flux_minus[-1], lw=1.5, label=f'total')
lines.append(line)

ax.set_xlabel('time (µs)')
ax.set_ylabel('$log_{10}$(particles/s)')
ax.set_title('Cusp loss rate vs time')

labels = [h.get_label() for h in lines]
ax_text.legend(lines, labels)

add_external_parameter_box(fig, static_params, dx=-0.4, dy=-0.2)
plt.tight_layout()
plt.savefig(f'diags/cusp_loss_total.png', dpi=150)
plt.show()

# ax.plot(times, np.log10(flux_minus), color='steelblue', lw=1.5, label='leaving')
# # ax.plot(times, np.log10(flux_plus), color='red', lw=1.5, label='returning')
# ax.set_xlabel('time (µs)')
# ax.set_ylabel('$log_{10}$(particles/s)')
# ax.set_title('Cusp loss rate vs time')

# total_loss = flux_minus

# # ── Total loss rate vs time ───────────────────────────────────────
# fig, (ax, ax_text) = plt.subplots(1, 2, figsize=(8, 4), width_ratios=(3, 1))
# ax_text.axis('off')
# ax.plot(times, np.log10(total_loss), color='steelblue', lw=1.5, label='leaving')
# ax.plot(times, np.log10(flux_plus), color='red', lw=1.5, label='returning')
# ax.set_xlabel('time (µs)')
# ax.set_ylabel('$log_{10}$(particles/s)')
# ax.set_title('Cusp loss rate vs time')
# ax.legend()
# add_external_parameter_box(fig, static_params)
# plt.tight_layout()
# plt.savefig(f'diags/cusp_loss_total.png', dpi=150)
# plt.show()

# Streamplot on top of beta

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import scipy.constants as sc
from scipy.ndimage import gaussian_filter
from openpmd_viewer import OpenPMDTimeSeries

MU0 = sc.mu_0

# ── Configuration ────────────────────────────────────────────────
save_path  = "diags"

# ── Merged Precompute Loop ───────────────────────────────────────
#beta_frames  = []
# B_streamplots = []
# xs = zs = None

# for k, it in enumerate(iterations[1:]):
#     print(f"  Loading {k+1}/{len(iterations)-1}", end='\r')

#     Bx, info = series_f.get_field('B', coord='x', iteration=it, slice_across='y')
#     By, _    = series_f.get_field('B', coord='y', iteration=it, slice_across='y')
#     Bz, _    = series_f.get_field('B', coord='z', iteration=it, slice_across='y')
#     rho,    _ = series_f.get_field('rho_stream_i',     iteration=it, slice_across='y')
#     rho_bg, _ = series_f.get_field('rho_background_i', iteration=it, slice_across='y')

#     if xs is None:
#         xs, zs = info.x, info.z

#     # Beta frame
#     B2     = np.maximum(Bx**2 + By**2 + Bz**2, 1e-12)
#     n      = np.abs(rho + rho_bg) / sc.e
#     Ti_J   = Ti_eV * sc.eV
#     beta_th = n * Ti_J / (B2 / (2 * MU0))
#     #beta_frames.append(gaussian_filter(np.log10(np.maximum(beta_th, 1e-3)), sigma=1))

#     # Streamplot frame (Bx, Bz in the xz plane)
#     B_mag = np.sqrt(Bx**2 + Bz**2)
#     B_streamplots.append((Bx, Bz, B_mag))

# print("\nDone.")

# ── Helper: remove streamplot artists ───────────────────────────
def _clear_streamplot(sp):
    sp.lines.remove()


# ── Figure Setup ─────────────────────────────────────────────────
fig, (ax, ax_text) = plt.subplots(1, 2, figsize=(10, 5), width_ratios=(2, 1))
ax_text.axis('off')
ax2 = ax.inset_axes([0, 0, 1, 1])
ax2.set_axis_off()
ax2.patch.set_alpha(0)

# Layer 0: beta heatmap
im = ax.imshow(
    beta_frames[0], origin='lower',
    extent=[xs[0], xs[-1], zs[0], zs[-1]],
    vmin=-3, vmax=3, cmap='RdBu_r', aspect='equal', zorder=0
)

# Layer 1: beta=1 contour
contour_handle = [ax.contour(
    xs, zs, beta_frames[0], levels=[0.0],
    colors='k', linewidths=1.0, zorder=1
)]

vmin_stream = np.min([s[2] for s in B_streamplots])
vmax_stream = np.max([s[2] for s in B_streamplots])
norm = mcolors.Normalize(vmin=vmin_stream, vmax=vmax_stream)  # Explicitly enforces 0 as white across all frames

# Layer 2: B-field streamplot on top
stream_handle = [ax2.streamplot(
    xs, zs,
    B_streamplots[0][0], B_streamplots[0][1], norm=norm,
    color=B_streamplots[0][2], cmap='hot', density=1.2, linewidth=0.8, zorder=2,
)]

plt.colorbar(im, ax=ax, label=r'$log_{10}$ $\beta$', pad=0.1)
plt.colorbar(stream_handle[0].lines, ax=ax, label='|B|', pad=0.1)
ax.set_xlabel('x (m)')
ax.set_ylabel('z (m)')
ax.legend(loc='upper right', fontsize=8)
title = ax.set_title('')

add_external_parameter_box(fig, static_params)

plt.savefig('test.png')
plt.show()

# ── Animation Update ─────────────────────────────────────────────
def update(k):
    it = iterations[k]
    ax2.cla()
    ax2.set_axis_off()
    ax2.patch.set_alpha(0)
    ax2.set_xlim(ax.get_xlim())
    ax2.set_ylim(ax.get_ylim())

    # Update beta heatmap
    im.set_data(beta_frames[k])

    # Update beta=1 contour
    contour_handle[0].remove()
    contour_handle[0] = ax.contour(
        xs, zs, beta_frames[k], levels=[0.0],
        colors='k', linewidths=1.0, zorder=1
    )

    stream_handle[0] = ax2.streamplot(
        xs, zs,
        B_streamplots[k][0], B_streamplots[k][1],
        color=B_streamplots[k][2], cmap='hot',  density=1.2, linewidth=0.8, zorder=2,
    )

    t_us = series_f.t[k] * 1e6
    title.set_text(f'β_th  step {it}  t = {t_us:.2f} µs')
    return [im]

# ── Save ─────────────────────────────────────────────────────────
ani = animation.FuncAnimation(
    fig, update, frames=len(iterations), interval=100
)
ani.save(f'{save_path}/beta_stream_overlay.mp4', writer='ffmpeg', fps=10, dpi=300)
plt.close()
print(f"Saved: {save_path}/beta_stream_overlay.mp4")

# Bx lineout 

In [ ]:
# ── Precompute ───────────────────────────────────────────────────
B_lineouts = []
xs_line    = None

for k, it in enumerate(iterations):
    print(f"  Loading {k+1}/{len(iterations)}", end='\r')

    Bx, info = series_f.get_field('B', coord='x', iteration=it, slice_across=['y', 'z'])
    By, _    = series_f.get_field('B', coord='y', iteration=it, slice_across=['y', 'z'])
    Bz, _    = series_f.get_field('B', coord='z', iteration=it, slice_across=['y', 'z'])

    if xs_line is None:
        xs_line = info.x

    # B_mag = np.sqrt(Bx**2 + By**2 + Bz**2)
    # B_lineouts.append(gaussian_filter(B_mag, sigma=1))
    B_lineouts.append(Bx)

print("\nDone.")

# ── Animate ──────────────────────────────────────────────────────
B_max = max(b.max() for b in B_lineouts)
B_min = min(b.min() for b in B_lineouts)

fig, (ax, ax_text) = plt.subplots(1, 2, figsize=(8, 4), width_ratios=(3, 1), layout='constrained')
ax_text.axis('off')
line,  = ax.plot(xs_line, B_lineouts[0], color='steelblue', lw=2)
ax.axhline(0, color='k', lw=0.5, ls='--')
ax.set_ylim(B_min * 1.1, B_max * 1.1)
ax.set_xlabel('x (m)')
ax.set_ylabel('Bx (T)')
title = ax.set_title('')

add_external_parameter_box(fig, static_params)

def update(k):
    line.set_ydata(B_lineouts[k])
    t_us = series_f.t[k] * 1e6
    title.set_text(f'Bx lineout  step {iterations[k]}  t = {t_us:.2f} µs')
    return [line]

ani = animation.FuncAnimation(fig, update, frames=len(iterations), interval=100)
ani.save(f'{save_path}/Bx_lineout_timelapse.mp4', writer='ffmpeg', fps=10, dpi=150)
plt.close()
print(f"Saved: {save_path}/Bx_lineout_timelapse.mp4")

# Bx for different r

In [ ]:
# ── Precompute ───────────────────────────────────────────────────
rs = np.linspace(0, 0.5, 6)
print(rs)
Bxr = {r: [] for r in rs}

for k, it in enumerate(iterations):
    print(f"  Loading {k+1}/{len(iterations)}", end='\r')

    for r in rs:

        Bx, info = series_f.get_field('B', coord='x', iteration=it, slice_across=['y', 'z'], slice_relative_position=[0, r / 1.5])
        By, _    = series_f.get_field('B', coord='y', iteration=it, slice_across=['y', 'z'], slice_relative_position=[0, r / 1.5])
        Bz, _    = series_f.get_field('B', coord='z', iteration=it, slice_across=['y', 'z'], slice_relative_position=[0, r / 1.5])

        xs_line = info.x

        Bxr[r].append(Bx)

print("\nDone.")

fig, (ax, ax_text) = plt.subplots(1, 2, figsize=(8, 4), width_ratios=(3, 1), layout='constrained')
ax_text.axis('off')

ax.axhline(0, color='k', lw=0.5, ls='--')
# ax.set_ylim(B_min * 1.1, B_max * 1.1)
ax.set_xlabel('x (m)')
ax.set_ylabel('Bx (T)')
title = ax.set_title('')

B_min = min(min(b.min() for b in Bxr[r]) for r in rs)
B_max = max(max(b.max() for b in Bxr[r]) for r in rs)

ax.set_ylim(B_min * 1.1, B_max * 1.1)

lines = []

for r in rs:
    # ── Animate ──────────────────────────────────────────────────────
    # B_max = max(b.max() for b in Bxr[r])
    # B_min = min(b.min() for b in Bxr[r])
    line,  = ax.plot(xs_line, Bxr[r][0], label=f"{r:.1f}")
    lines.append(line)

ax.legend()

add_external_parameter_box(fig, static_params)

plt.show()

def update(k):
    for i, r in enumerate(rs):
        lines[i].set_ydata(Bxr[r][k])
    t_us = series_f.t[k] * 1e6
    title.set_text(f'Bx lineout @ different r from coil center, step {iterations[k]}, t = {t_us:.2f} µs')
    return [line]

ani = animation.FuncAnimation(fig, update, frames=len(iterations), interval=100)
ani.save(f'{save_path}/Bxr_lineout_timelapse.mp4', writer='ffmpeg', fps=10, dpi=150)
plt.close()
print(f"Saved: {save_path}/Bxr_lineout_timelapse.mp4")

# Cusp losses time-synced

In [ ]:
data   = np.load(f'{save_path}/cusp_flux.npz', allow_pickle=True)
times  = data['times'] * 1e6          # convert to µs
flux_minus = data['flux_minus']               # shape (n_steps, 6)
flux_plus = data['flux_plus']
#flux_minus_upstream = data['flux_minus_upstream']
#labels = data['face_labels']

total_loss = flux_minus.T[0]
t_k = [None]
text = [None]
# ── Total loss rate vs time ───────────────────────────────────────
fig, (ax, ax_text) = plt.subplots(1, 2, figsize=(8, 4), width_ratios=(3, 1))
ax_text.axis('off')
ax.plot(times, total_loss, color='steelblue', lw=1.5, label='Flux through coil @ x = 0.0')
#ax.plot(times, flux_minus_upstream, color='c', lw=1.5, label='Upstream Flux @ x = 2.0')
#ax.axhline(np.mean(flux_minus_upstream[20:]), label="Upstream flow average", color='g', linestyle='--')
#ax.text(0, np.mean(flux_minus_upstream[20:]), f'{np.mean(flux_minus_upstream[20:]):.2e}')
diag_times = series_f.t * 1e6
print(len(diag_times), len(total_loss))
t_k[0] = ax.axvline(diag_times[0], color='red')
ax.set_xlabel('time (µs)')
ax.set_ylabel('particles lost per step')
ax.set_title('Cusp loss rate vs time')
text[0] = ax.text(diag_times[0], total_loss[0], f"{total_loss[0]:.2e}")
# ax.legend(loc='upper right')
add_external_parameter_box(fig, static_params)
plt.tight_layout()

def update(k):
    it = iterations[k]
    t = series_f.t[k] * 1e6
    title = f"Cusp loss rate it: {it} t: {t:.2f} us"
    ax.set_title(title)
    t_k[0].remove()
    t_k[0] = ax.axvline(diag_times[k], color='red')
    text[0].remove()
    xlim = ax.get_xlim()
    frac = (diag_times[k] - xlim[0]) / (xlim[1] - xlim[0])
    ha = 'right' if frac > 0.7 else 'left'
    offset = -0.02 * (xlim[1] - xlim[0]) if ha == 'right' else 0.02 * (xlim[1] - xlim[0])
    try:
        idx = np.searchsorted(times, diag_times[k])
        text[0] = ax.text(diag_times[k] + offset, total_loss[idx], f"{total_loss[idx]:.2e}", ha=ha)
    except:
        text[0] = ax.text(diag_times[-1] + offset, total_loss[-1], f"{total_loss[-1]:.2e}", ha=ha)

ani = animation.FuncAnimation(fig, update, frames=len(iterations), interval=100)
ani.save(f'{save_path}/cusp_loss_time_synced.mp4', writer='ffmpeg', fps=10, dpi=150)
plt.close()
print(f"Saved: {save_path}/cusp_loss_time_synced.mp4")

# Density over B-lines

In [ ]:
MU0 = sc.mu_0

# ── Merged Precompute Loop ───────────────────────────────────────
density_frames  = []
# B_streamplots = []
# xs = zs = None
w = series_p.get_particle(var_list=['w'], iteration=iterations[5])
for k, it in enumerate(iterations):
    print(f"  Loading {k+1}/{len(iterations)}", end='\r')

    Bx, info = series_f.get_field('B', coord='x', iteration=it, slice_across='y')
    By, _    = series_f.get_field('B', coord='y', iteration=it, slice_across='y')
    Bz, _    = series_f.get_field('B', coord='z', iteration=it, slice_across='y')
    rho,    _ = series_f.get_field('rho_stream_i',     iteration=it, slice_across='y')
    
    xs, zs = info.x, info.z

    # Density
    density_frames.append(np.log10((rho + 1e-12) / sc.elementary_charge))

print("\nDone.")

# ── Helper: remove streamplot artists ───────────────────────────
def _clear_streamplot(sp):
    sp.lines.remove()


# ── Figure Setup ─────────────────────────────────────────────────
fig, (ax, ax_text) = plt.subplots(1, 2, figsize=(10, 5), width_ratios=(2, 1))
ax_text.axis('off')
ax2 = ax.inset_axes([0, 0, 1, 1])
ax2.set_axis_off()
ax2.patch.set_alpha(0)

vmin_density = np.min([d for d in density_frames])
vmax_density = np.max([d for d in density_frames])

vmin_stream = np.min([s[2] for s in B_streamplots])
vmax_stream = np.max([s[2] for s in B_streamplots])
norm = mcolors.Normalize(vmin=vmin_stream, vmax=vmax_stream)

# Layer 0: beta heatmap
im = ax.imshow(
    density_frames[0], origin='lower',
    extent=[xs[0], xs[-1], zs[0], zs[-1]],
    vmin=vmin_density, vmax=vmax_density, cmap='plasma', aspect='equal', zorder=0
)

# Layer 2: B-field streamplot on top
stream_handle = [ax2.streamplot(
    xs, zs,
    B_streamplots[0][0], B_streamplots[0][1], norm = norm,
    color=B_streamplots[0][2], cmap='hot', density=1.2, linewidth=0.8, zorder=2,
)]

cb_density = plt.colorbar(im, ax=ax, label=r'$log_{10}$ $n$', pad=0.15)
cb_stream = plt.colorbar(stream_handle[0].lines, ax=ax, label='|B|', pad=0.1)
ax.set_xlabel('x (m)')
ax.set_ylabel('z (m)')
ax.legend(loc='upper right', fontsize=8)
title = ax.set_title('')

add_external_parameter_box(fig, static_params)
plt.savefig('test.png')
plt.show()

# ── Animation Update ─────────────────────────────────────────────
def update(k):
    it = iterations[k]
    ax2.cla()
    ax2.set_axis_off()
    ax2.patch.set_alpha(0)
    ax2.set_xlim(ax.get_xlim())
    ax2.set_ylim(ax.get_ylim())

    # Update density map
    im.set_data(density_frames[k])
    im.set_clim(np.min(density_frames[k]), np.max(density_frames[k]))

    stream_handle[0] = ax2.streamplot(
        xs, zs,
        B_streamplots[k][0], B_streamplots[k][1],
        color=B_streamplots[k][2], cmap='hot',  density=1.2, linewidth=0.8, zorder=2,
    )

    t_us = series_f.t[k] * 1e6
    title.set_text(r'$n$' + f' step {it}  t = {t_us:.2f} µs')
    return [im]

# ── Save ─────────────────────────────────────────────────────────
ani = animation.FuncAnimation(
    fig, update, frames=len(iterations), interval=100
)
ani.save(f'{save_path}/density_stream_overlay.mp4', writer='ffmpeg', fps=10, dpi=300)
plt.close()
print(f"Saved: {save_path}/density_stream_overlay.mp4")

# Current Jy on xz-plane @ y = 0

Note that $x \times z = -y$ and hence $sign(J_y) < 0 \rightarrow \odot$ direction, while $sign(J_y) > 0 \rightarrow \oplus$ direction

In [ ]:
J_s = []
xs, zs = None, None
for it in iterations:
    print(f"{it} / {iterations[-1]}", end="\r")
    Jx_ion, info = series_f.get_field(field='j', coord='x', slice_across='y', iteration=it)
    Jy_ion, info = series_f.get_field(field='j', coord='y', slice_across='y', iteration=it)
    Jz_ion, info = series_f.get_field(field='j', coord='z', slice_across='y', iteration=it)
    Jx_ele, _ = series_f.get_field(field='j_displacement', coord='x', slice_across='y', iteration=it)
    Jy_ele, _ = series_f.get_field(field='j_displacement', coord='y', slice_across='y', iteration=it)
    Jz_ele, _ = series_f.get_field(field='j_displacement', coord='z', slice_across='y', iteration=it)

    Jx = Jx_ion + Jx_ele
    Jy = Jy_ion + Jy_ele 
    Jz = Jz_ion + Jz_ele

    J_mag = np.sqrt(Jx**2 + Jy**2 + Jz**2)

    if xs is None:
        xs = info.x
        zs = info.z

    J_s.append([Jx, Jy, Jz, J_mag])

fig, (ax, ax_text) = plt.subplots(1, 2, figsize=(10, 5), width_ratios=(2, 1))
ax_text.axis('off')
vmin = np.min([s[1] for s in J_s])
vmax = np.max([s[1] for s in J_s])
norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
im = ax.imshow(J_s[0][1], origin='lower',
               extent=[xs[0], xs[-1], zs[0], zs[-1]],
               cmap='RdBu_r', aspect='equal', vmin=vmin, vmax=vmax)


plt.colorbar(im, ax=ax, label='$J_{y}$')
ax.set_xlabel('x (m)')
ax.set_ylabel('z (m)')
title = ax.set_title('J_y on xz-plane')

add_external_parameter_box(fig, static_params)

ax_text.text(
    0.1, 0.9, r'$-y: \odot$' + '\n' + r'$+y: \oplus$', fontsize=15, weight='medium',
    verticalalignment='center', horizontalalignment='left', transform=fig.axes[1].transAxes,
    bbox=dict(boxstyle="round,pad=0.6", facecolor='#f8f9fa', linewidth=1.5)
)

plt.savefig('test.png')
plt.show()

def update(k):
    it = iterations[k]
    im.set_data(J_s[k][1])

    t_us = series_f.t[k] * 1e6
    title.set_text(f'Jy on xz-plane  step {it}  t = {t_us:.2f} µs')
    return [im]

ani = animation.FuncAnimation(fig, update, frames=len(iterations), interval=100)
ani.save(f'{save_path}/Jy_xz_plane_timelapse.mp4', writer='ffmpeg', fps=10, dpi=300)
plt.close()
print(f"Saved: {save_path}/Jy_xz_plane_timelapse.mp4")

# $J_\theta(y, z)$

## 2D map of azimuthal (ring-current) component of plasma currents

### What it represents
- Each pixel is local current vector, tangential direction around x-axis
    - Circulating like a ring around a coil

### Azimuthal instead of cartesian
- Isolation of circular motion about the x-axis (to identify diamagnetic effects of gyromotion)

### +/- signs
- Positive: same rotation as coil (paramagnetic)
- Negative: opposing (diamagnetic)
- Where are the coil/plasma current standoffs occurring 

### Separating protons, electrons
- Influenced differently, with differing diamagnetic effects
- Understand fluid vs kinetic contributions

### Limitations
- Single x-value (@ x = 0)

In [ ]:
J_thetas = []

J_thetas = {
    'e': [],
    'p': [],
    't': []
}


for it in iterations:
    print(f"Processed: {it} / {iterations[-1]}", end='\r')
    Jx_ion, info = series_f.get_field(field='j', coord='x', slice_across='x', iteration=it)
    Jy_ion, info = series_f.get_field(field='j', coord='y', slice_across='x', iteration=it)
    Jz_ion, info = series_f.get_field(field='j', coord='z', slice_across='x', iteration=it)
    Jx_ele, _ = series_f.get_field(field='j_displacement', coord='x', slice_across='x', iteration=it)
    Jy_ele, _ = series_f.get_field(field='j_displacement', coord='y', slice_across='x', iteration=it)
    Jz_ele, _ = series_f.get_field(field='j_displacement', coord='z', slice_across='x', iteration=it)

    Jx = Jx_ion + Jx_ele
    Jy = Jy_ion + Jy_ele 
    Jz = Jz_ion + Jz_ele

    ys, zs = info.y, info.z

    # meshgrid ordering matches existing field array shape: rows=z, cols=y
    Z, Y = np.meshgrid(zs, ys, indexing='ij')
    r = np.sqrt(Y**2 + Z**2)

    # mask the on-axis singularity (r->0) rather than letting it blow up
    #r_safe = np.where(r < 2 * (ys[1] - ys[0]), np.nan, r)  # ~2 cells around axis

    J_theta_tot = (Jz * Y - Jy * Z) / r
    J_theta_e = (Jz_ele * Y - Jy_ele * Z) / r
    J_theta_p = (Jz_ion * Y - Jy_ion * Z) / r

    J_thetas['t'].append(J_theta_tot)
    J_thetas['e'].append(J_theta_e)
    J_thetas['p'].append(J_theta_p)

species_to_title = {
    'e': r'$J_e$',
    'p': r'$J_p$',
    't': r'$J_e + J_p$'
}

for species in ['t', 'e', 'p']:

    fig, (ax, ax_text) = plt.subplots(1, 2, figsize=(10, 5), width_ratios=(2, 1))
    ax_text.axis('off')
    # ax_text.text(
    #     0.1, 0.9, r'$+y: \odot$' + '\n' + r'$-y: \oplus$', fontsize=15, weight='medium',
    #     verticalalignment='center', horizontalalignment='left', transform=fig.axes[1].transAxes,
    #     bbox=dict(boxstyle="round,pad=0.6", linewidth=1.5)
    # )
    # fixed vmin and vmax
    vmax = np.nanmax([np.abs(i) for i in J_thetas[species]])
    vmin = -vmax
    im.set_clim(vmin=-vmax, vmax=vmax)

    im = ax.imshow(J_thetas[species][0], origin='lower',
                extent=[ys[0], ys[-1], zs[0], zs[-1]],
                cmap='RdBu_r', aspect='equal',
                vmin = vmin, vmax=vmax)

    plt.colorbar(im, ax=ax, label=r'$J_{\theta}$')
    ax.set_xlabel('y (m)')
    ax.set_ylabel('z (m)')
    title = ax.set_title(r'$J_{\theta}$ on yz-plane' + f', species: {species_to_title[species]}')

    add_external_parameter_box(fig, static_params)
    plt.savefig(f'test_{species}.png')
    plt.show()

    def update(k):
        it = iterations[k]
        im.set_data(J_thetas[species][k])
        vmin = np.min(J_thetas[species][k])
        vmax = np.max(J_thetas[species][k])

        t_us = series_f.t[k] * 1e6
        title.set_text(r'$J_{\theta}$ on yz-plane, ' + f'species: {species} it {it:.3f} t = {t_us:.2f} us')
        return [im]

    ani = animation.FuncAnimation(fig, update, frames=len(iterations), interval=100)
    ani.save(f'{save_path}/Jtheta_{species}_yz_plane_timelapse.mp4', writer='ffmpeg', fps=10, dpi=300)
    plt.close()
    print(f"Saved: {save_path}/Jtheta_{species}_yz_plane_timelapse.mp4")

# J streamplot on yz-plane

## Capturing:
- Closed loops about x -> azimuthal (ring currents) - diamagnetic/paramagnetic flow shapes
- Spirals in/out/radially -> compression/expansion

In [ ]:
ys, zs, r = None, None, None
Js = []
for it in iterations:
    print(f"{it}/{iterations[-1]}", end="\r")
    Jx_ion, info = series_f.get_field(field='j', coord='x', slice_across='x', iteration=it)
    Jy_ion, info = series_f.get_field(field='j', coord='y', slice_across='x', iteration=it)
    Jz_ion, info = series_f.get_field(field='j', coord='z', slice_across='x', iteration=it)
    Jx_ele, _ = series_f.get_field(field='j_displacement', coord='x', slice_across='x', iteration=it)
    Jy_ele, _ = series_f.get_field(field='j_displacement', coord='y', slice_across='x', iteration=it)
    Jz_ele, _ = series_f.get_field(field='j_displacement', coord='z', slice_across='x', iteration=it)

    Jx = Jx_ion + Jx_ele
    Jy = Jy_ion + Jy_ele 
    Jz = Jz_ion + Jz_ele

    J_mag = np.sqrt(Jy**2 + Jz**2)

    if ys is None:
        ys = info.y
        zs = info.z
        # meshgrid built once — reused every iteration since grid is static
        Z, Y = np.meshgrid(zs, ys, indexing='ij')
        r = np.sqrt(Y**2 + Z**2)

    # signed azimuthal current: + = same sense as coil (paramagnetic),
    # - = opposing sense (diamagnetic) — see established sign convention
    J_theta = (Jz * Y - Jy * Z) / r

    Js.append([Jy, Jz, J_theta])

# global symmetric color scale so 0 always maps to white and frames are
# comparable to each other (same reasoning as the J_theta imshow plots)

vmax = np.nanmax(np.abs([j[2] for j in Js]))
norm = mcolors.Normalize(vmin=-vmax, vmax=vmax)  # Explicitly enforces 0 as white across all frames

fig, (ax, ax_text) = plt.subplots(1, 2, figsize=(10, 5), layout='constrained', width_ratios=[2, 1])
ax_text.axis('off')
im = ax.streamplot(
    x=ys,
    y=zs,
    u=Js[0][0],
    v=Js[0][1],
    color=Js[0][2],
    norm=norm,
    cmap="RdBu_r", density=2.0, linewidth=1,
    broken_streamlines=False,
)

cbar = fig.colorbar(im.lines, ax=ax, label=r'$J_\theta$')

ax.set_xlabel('y (m)')
ax.set_ylabel('z (m)')
# force fixed extent every frame — don't rely on streamplot's autoscale
ax.set_xlim(ys[0], ys[-1])
ax.set_ylim(zs[0], zs[-1])
ax.set_aspect('equal')  # streamplot resets this too on cla()
title = ax.set_title(f'J step 0 t = {series_f.t[0]*1e6} µs')

add_external_parameter_box(params=static_params, ax=ax_text)

plt.savefig('example_fig.png', )
plt.show()

def update(k):
    print(f"{k} / {len(iterations)}", end='\r')
    it = iterations[k]
    ax.cla()
    im = ax.streamplot(
        x=ys,
        y=zs,
        u=Js[k][0],
        v=Js[k][1],
        color=Js[k][2],
        norm=norm,
        cmap="RdBu_r", density=2.0, linewidth=1
    )
    # force fixed extent every frame — don't rely on streamplot's autoscale
    ax.set_xlim(ys[0], ys[-1])
    ax.set_ylim(zs[0], zs[-1])
    ax.set_aspect('equal')  # streamplot resets this too on cla()
    t_us = series_f.t[k] * 1e6
    ax.set_title(f'J  step {it}  t = {t_us:.2f} µs')
    ax.set_xlabel('y (m)')
    ax.set_ylabel('z (m)')
    return [im]

ani = animation.FuncAnimation(fig, update, frames=len(iterations), interval=100)
ani.save(f'{save_path}/J_stream_timelapse.mp4', writer='ffmpeg', fps=10, dpi=300)
plt.close()
print(f"Saved: {save_path}/J_stream_timelapse.mp4")

# E Field

In [ ]:
E_stuff = []

for it in iterations:

    print(f"Complete: {it / iterations[-1]:.2%}", end='\r')

    Ex, info = series_f.get_field(field='E', coord='x', iteration=it, slice_across='y')
    Ey, _ = series_f.get_field(field='E', coord='y', iteration=it, slice_across='y')
    Ez, _ = series_f.get_field(field='E', coord='z', iteration=it, slice_across='y')

    E_mag = np.sqrt(Ex**2 + Ey**2 + Ez**2)

    E_stuff.append((Ex, Ez, E_mag))

# Animate
E_max = max(e[2].max() for e in E_stuff)

fig, (ax, ax_text) = plt.subplots(1, 2, figsize=(10, 5), layout='constrained', width_ratios=[2, 1])
ax_text.axis('off')
ax.set_aspect('equal')
vmax = np.max([e[2] for e in E_stuff])
vmin = np.min([e[2] for e in E_stuff])
norm = mcolors.Normalize(vmin=vmin, vmax=vmax)  # Explicitly enforces 0 as white across all frames
im = ax.streamplot(
    x=info.x,
    y=info.z,
    u=E_stuff[0][0],
    v=E_stuff[0][1],
    color=E_stuff[0][2], norm=norm,
    cmap="viridis", density=2.0, linewidth=1,
    broken_streamlines=True,
)

ax.set_xlim((-cfg.L, cfg.L))
ax.set_ylim((-cfg.L, cfg.L))

cbar = fig.colorbar(im.lines, ax=ax, label=r'$|E|$')

ax.set_xlabel('x (m)')
ax.set_ylabel('z (m)')
title = ax.set_title('Iteration 0')

add_external_parameter_box(fig, static_params, dx=5)

plt.savefig('test.png')
plt.show()

def update(k):
    print(f"On iteration: {k}", end='\r')
    it = iterations[k]
    ax.cla()
    ax.set_xlim((-cfg.L, cfg.L))
    ax.set_ylim((-cfg.L, cfg.L))
    im = ax.streamplot(
        x=info.x,
        y=info.z,
        u=E_stuff[k][0],
        v=E_stuff[k][1],
        color=E_stuff[k][2],
        cmap="viridis", density=2.0, linewidth=1
    )
    t_us = series_f.t[k] * 1e6
    ax.set_title(f'|E|  step {it}  t = {t_us:.2f} µs')
    ax.set_xlabel('x (m)')
    ax.set_ylabel('z (m)')
    return [im]

ani = animation.FuncAnimation(fig, update, frames=len(iterations), interval=100)
ani.save(f'{save_path}/E_stream_timelapse.mp4', writer='ffmpeg', fps=10, dpi=300)
plt.close()
print(f"Saved: {save_path}/E_stream_timelapse.mp4")

# Analyzing E terms from WarpX solver

In [ ]:
e = sc.e
eta = cfg.eta_bg
eta_h = cfg.eta_H

it = iterations[99]

Jx_i, info = series_f.get_field(field='j', coord='x', iteration=it) # (64, 64, 64)
Jy_i, info = series_f.get_field(field='j', coord='y', iteration=it)
Jz_i, info = series_f.get_field(field='j', coord='z', iteration=it)
Jx_e, _ = series_f.get_field(field='j_displacement', coord='x', iteration=it) # (64, 64, 64)
Jy_e, _ = series_f.get_field(field='j_displacement', coord='y', iteration=it)
Jz_e, _ = series_f.get_field(field='j_displacement', coord='z', iteration=it)

Jx = Jx_i + Jx_e
Jy = Jy_i + Jy_e 
Jz = Jz_i + Jz_e

Bx, _ = series_f.get_field(field='B', coord='x', iteration=it) # (64, 64, 64)
By, _ = series_f.get_field(field='B', coord='y', iteration=it)
Bz, _ = series_f.get_field(field='B', coord='z', iteration=it)

rho, _ = series_f.get_field(field='rho', iteration=it) # (64, 64, 64)
n_i = n_e = rho / sc.e

Pe = cfg.n_stream * cfg.T_i_eV * sc.e * (n_e / cfg.n_stream)**(5/3)

# Load WarpX's own E-field diagnostic at the same iteration
Ex_diag, info = series_f.get_field(field='E', coord='x', iteration=it)
Ey_diag, _    = series_f.get_field(field='E', coord='y', iteration=it)
Ez_diag, _    = series_f.get_field(field='E', coord='z', iteration=it)

# Grid spacing for gradient/laplacian (from openPMD metadata)
dx, dy, dz = info.x[1] - info.x[0], info.y[1] - info.y[0], info.z[1] - info.z[0]  # confirm units/order match array axes

# --- Hall + pressure term ---
Jx_e_arr, Jy_e_arr, Jz_e_arr = Jx_e, Jy_e, Jz_e  # J_e = j_displacement, per WarpX docs
JxB_x = Jy_e_arr*Bz - Jz_e_arr*By
JxB_y = Jz_e_arr*Bx - Jx_e_arr*Bz
JxB_z = Jx_e_arr*By - Jy_e_arr*Bx

dPe_dx, dPe_dy, dPe_dz = np.gradient(Pe, dx, dy, dz)

en = e * n_e  # avoid /0 near cusp null — mask below

# Ex_hp = (JxB_x - dPe_dx) / en
# Ey_hp = (JxB_y - dPe_dy) / en
# Ez_hp = (JxB_z - dPe_dz) / en
rho_floor = cfg.n_floor * e   # hybrid_pic_model.n_floor, matches source: m_n_floor * q_e
vacuum_mask = rho < rho_floor  # rho = n_e * e, same array
Ex_hp = np.where(vacuum_mask, 0.0, (JxB_x - dPe_dx) / rho)
Ey_hp = np.where(vacuum_mask, 0.0, (JxB_y - dPe_dy) / rho)
Ez_hp = np.where(vacuum_mask, 0.0, (JxB_z - dPe_dz) / rho)

# --- Resistive term (J_ext excluded — confirmed from source) ---
Ex_resist = eta * Jx
Ey_resist = eta * Jy
Ez_resist = eta * Jz

# --- Hyper-resistive term (skip if eta_h = 0 in your config) ---
def laplacian(F, dx, dy, dz):
    grad_x, grad_y, grad_z = np.gradient(F, dx, dy, dz, edge_order=2)
    print(grad_x.shape, grad_y.shape, grad_z.shape)
    lap_x = np.gradient(grad_x, dx, axis=0, edge_order=2)
    lap_y = np.gradient(grad_y, dy, axis=1, edge_order=2)
    lap_z = np.gradient(grad_z, dz, axis=2, edge_order=2)

    lap = lap_x + lap_y + lap_z

    print(lap.shape)

    return lap

def laplacian_nodal(F, dx, dy, dz):
    """Matches WarpX CartesianNodalAlgorithm::Dxx+Dyy+Dzz (3-pt central 2nd derivative per axis)."""
    d2Fdx2 = (np.roll(F, -1, axis=0) - 2*F + np.roll(F, 1, axis=0)) / dx**2
    d2Fdy2 = (np.roll(F, -1, axis=1) - 2*F + np.roll(F, 1, axis=1)) / dy**2
    d2Fdz2 = (np.roll(F, -1, axis=2) - 2*F + np.roll(F, 1, axis=2)) / dz**2
    return d2Fdx2 + d2Fdy2 + d2Fdz2

Ex_hyper = -eta_h * laplacian(Jx, dx, dy, dz)
Ey_hyper = -eta_h * laplacian(Jy, dx, dy, dz)
Ez_hyper = -eta_h * laplacian(Jz, dx, dy, dz)

# --- Reconstructed total ---
Ex_recon = Ex_hp + Ex_resist + Ex_hyper
Ey_recon = Ey_hp + Ey_resist + Ey_hyper
Ez_recon = Ez_hp + Ez_resist + Ez_hyper

resid = np.abs(np.concatenate([(Ex_recon-Ex_diag).ravel(),
                                (Ey_recon-Ey_diag).ravel(),
                                (Ez_recon-Ez_diag).ravel()]))
ref   = np.abs(np.concatenate([Ex_diag.ravel(), Ey_diag.ravel(), Ez_diag.ravel()]))
print("median relative residual:", np.median(resid) / np.median(ref))

# pick an interior bulk-plasma index, well clear of cusp/floor
i0, j0, k0 = 32, 32, 40  # adjust to a location you know is unremarkable

print("E_diag:      ", Ex_diag[i0,j0,k0], Ey_diag[i0,j0,k0], Ez_diag[i0,j0,k0])
print("E_hp:        ", Ex_hp[i0,j0,k0], Ey_hp[i0,j0,k0], Ez_hp[i0,j0,k0])
print("E_resist:    ", Ex_resist[i0,j0,k0], Ey_resist[i0,j0,k0], Ez_resist[i0,j0,k0])
print("E_hyper:     ", Ex_hyper[i0,j0,k0], Ey_hyper[i0,j0,k0], Ez_hyper[i0,j0,k0])
print("E_recon sum: ", Ex_recon[i0,j0,k0], Ey_recon[i0,j0,k0], Ez_recon[i0,j0,k0])

print("dx, dy, dz:", dx, dy, dz)

# Cross-check against your deck's actual grid geometry (Lx/Ly/Lz over n_cell)
# e.g. from inputs_test file: geometry.prob_hi/prob_lo and amr.n_cell
Lx_expected = cfg.L * 2 / cfg.N   # pull actual values from your deck, not memory
Ly_expected = cfg.L * 2 / cfg.N
Lz_expected = cfg.L * 2 / cfg.N
print("expected dx,dy,dz:", Lx_expected, Ly_expected, Lz_expected)

print("len(info.x), array shape along axis 0:", len(info.x), Jx.shape[0])

# $E_{\parallel B}$

In [ ]:
E_pars = []

for it in iterations:
    print(f"It{it}", end='\r')

    # --- 2. E_parallel to B ---

    Bx, _ = series_f.get_field(field='B', coord='x', iteration=it) # (64, 64, 64)
    By, _ = series_f.get_field(field='B', coord='y', iteration=it)
    Bz, _ = series_f.get_field(field='B', coord='z', iteration=it)

    Ex_diag, info = series_f.get_field(field='E', coord='x', iteration=it)
    Ey_diag, _    = series_f.get_field(field='E', coord='y', iteration=it)
    Ez_diag, _    = series_f.get_field(field='E', coord='z', iteration=it)

    Bmag = np.sqrt(Bx**2 + By**2 + Bz**2)
    
    Bx_hat, By_hat, Bz_hat = Bx/Bmag, By/Bmag, Bz/Bmag   # watch div-by-zero at true nulls

    E_par = Ex_diag*Bx_hat + Ey_diag*By_hat + Ez_diag*Bz_hat  # scalar field, same shape as B

    E_pars.append(E_par[:, 32, :])

vmin = -np.max([np.abs(e).max() for e in E_par])
vmax = -vmin

fig, (ax, ax_text) = plt.subplots(1, 2, figsize=(10,5), width_ratios=(3, 1))
ax_text.axis('off')
im = ax.imshow(
    E_par[:, j0, :],   # slice through y=j0, same convention as your B slices
    extent=[info.x[0], info.x[-1], info.z[0], info.z[-1]], vmin=vmin, vmax=vmax,
    origin='lower', cmap='RdBu_r', aspect='auto'
)
ax.set_aspect('equal')
ax.set_xlabel('x (m)'); ax.set_ylabel('z (m)')
title = ax.set_title(f'$E_\\parallel$  t = {it*cfg.dt*1e6:.2f} μs')
fig.colorbar(im, label='$E_\\parallel$ (V/m)')

add_external_parameter_box(fig, static_params, dx=6)

plt.show()

def update(k):
    print(f"It{k}", end='\r')
    it = iterations[k]
    im.set_data(E_pars[k])

    t_us = series_f.t[k] * 1e6
    title.set_text(r'$E_{\parallel}$, ' + f' it {it:.3f} t = {t_us:.2f} us')
    return [im]

ani = animation.FuncAnimation(fig, update, frames=len(iterations), interval=100)
ani.save(f'{save_path}/E_par_timelapse.mp4', writer='ffmpeg', fps=10, dpi=300)
print(f"Saved: {save_path}/E_par_timelapse.mp4")

# E

In [ ]:
Es = []

for it in iterations:
    print(f"It{it}", end='\r')

    # --- 2. E_parallel to B ---

    Bx, _ = series_f.get_field(field='B', coord='x', iteration=it) # (64, 64, 64)
    By, _ = series_f.get_field(field='B', coord='y', iteration=it)
    Bz, _ = series_f.get_field(field='B', coord='z', iteration=it)

    Ex_diag, info = series_f.get_field(field='E', coord='x', iteration=it)
    Ey_diag, _    = series_f.get_field(field='E', coord='y', iteration=it)
    Ez_diag, _    = series_f.get_field(field='E', coord='z', iteration=it)

    E = np.sqrt(Ex_diag**2 + Ey_diag**2 + Ez_diag**2)

    # Bmag = np.sqrt(Bx**2 + By**2 + Bz**2)
    
    # Bx_hat, By_hat, Bz_hat = Bx/Bmag, By/Bmag, Bz/Bmag   # watch div-by-zero at true nulls

    # E_par = Ex_diag*Bx_hat + Ey_diag*By_hat + Ez_diag*Bz_hat  # scalar field, same shape as B

    # # quick sanity mask: exclude cells where |B| is too small for B_hat to be meaningful
    # Bmag_floor = 1e-3 * np.nanmax(Bmag)  # tune empirically, same spirit as your NGP coherence masking
    # E_par_masked = np.where(Bmag > Bmag_floor, E_par, np.nan)

    #vmax = np.nanpercentile(np.abs(E_par[:, j0, :]), 99)  # robust to outliers, avoid hard clipping
    Es.append(E[:, 32, :])

vmin = 0
vmax = np.max([np.abs(e).max() for e in Es])

fig, (ax, ax_text) = plt.subplots(1, 2, figsize=(10,5), width_ratios=(3, 1))
ax_text.axis('off')
ax.set_aspect('equal')
im = ax.imshow(
    E[:, j0, :],   # slice through y=j0, same convention as your B slices
    extent=[info.x[0], info.x[-1], info.z[0], info.z[-1]], vmin=vmin, vmax=vmax,
    origin='lower', cmap='plasma', aspect='auto'
)
ax.set_xlabel('x (m)'); ax.set_ylabel('z (m)')
title = ax.set_title(f'$|E|$  t = {it*cfg.dt*1e6:.2f} μs')
fig.colorbar(im, label='$|E|$ (V/m)')

add_external_parameter_box(fig, params=static_params, dx=6)

def update(k):
    print(f"It{k}", end='\r')
    it = iterations[k]
    im.set_data(Es[k])

    t_us = series_f.t[k] * 1e6
    title.set_text(r'$|E|$, ' + f' it {it:.3f} t = {t_us:.2f} us')
    return [im]

ani = animation.FuncAnimation(fig, update, frames=len(iterations), interval=100)
ani.save(f'{save_path}/E_mag_timelapse.mp4', writer='ffmpeg', fps=10, dpi=300)
print(f"Saved: {save_path}/E_mag_timelapse.mp4")

# E lineouts

# |E| along x

In [ ]:
lineouts = []

for it in iterations:
    print(f"It{it}", end='\r')
    Ex_diag, info = series_f.get_field(field='E', coord='x',slice_across=['y', 'z'], iteration=it)
    Ey_diag, _    = series_f.get_field(field='E', coord='y',slice_across=['y', 'z'], iteration=it)
    Ez_diag, _    = series_f.get_field(field='E', coord='z',slice_across=['y', 'z'], iteration=it)

    E = np.sqrt(Ex_diag**2 + Ey_diag**2 + Ez_diag**2)

    lineouts.append([E, Ex_diag, Ey_diag, Ez_diag])

ymax = np.max([e[0] for e in lineouts])

fig, (ax, ax_text) = plt.subplots(1, 2, figsize=(10, 5), width_ratios=(3, 1))

ax.set_ylim((0, ymax))

ax_text.axis('off')

lines = []

line1, = ax.plot(info.x, E, label=r"$|E|$")
lines.append(line1)
title = ax.set_title(r"$|E| along x @ z = 0$")
ax.set_xlabel(r"$x (m)$")
ax.set_ylabel(r"$|E| (V/m)$")

# line2, = ax.plot(info.x, Ex_diag, label='$E_x$')
# line3, = ax.plot(info.x, Ey_diag, label='$E_y$')
# line4, = ax.plot(info.x, Ez_diag, label='$E_z$')
# lines.append(line2)
# lines.append(line3)
# lines.append(line4)

# ax.legend()

add_external_parameter_box(fig, static_params)

plt.show()

def update(k):
    for i in range(len(lines)):
        lines[i].set_ydata(lineouts[k][i])
    it = iterations[k]
    t_us = series_f.t[k] * 1e6
    title.set_text(r'$|E|$, ' + f'along x @ y = z = 0, it {it:.3f} t = {t_us:.2f} us')

ani = animation.FuncAnimation(fig, update, frames=len(iterations), interval=100)
ani.save(f'{save_path}/E_lineout_timelapse.mp4', writer='ffmpeg', fps=10, dpi=300)
print(f"Saved: {save_path}/E_lineout_timelapse.mp4")

# $E_x$

In [ ]:
lineouts = []

for it in iterations:
    print(f"It{it}", end='\r')
    Ex_diag, info = series_f.get_field(field='E', coord='x',slice_across=['y', 'z'], iteration=it)
    Ey_diag, _    = series_f.get_field(field='E', coord='y',slice_across=['y', 'z'], iteration=it)
    Ez_diag, _    = series_f.get_field(field='E', coord='z',slice_across=['y', 'z'], iteration=it)

    E = np.sqrt(Ex_diag**2 + Ey_diag**2 + Ez_diag**2)

    lineouts.append([E, Ex_diag, Ey_diag, Ez_diag])

ymax = np.max([e[1] for e in lineouts])
ymin = np.min([e[1] for e in lineouts])

fig, (ax, ax_text) = plt.subplots(1, 2, figsize=(10, 5), width_ratios=(3, 1))

ax.set_ylim((ymin, ymax))

ax_text.axis('off')

lines = []

line1, = ax.plot(info.x, Ex_diag, label=r"$|E|$")
lines.append(line1)
title = ax.set_title(r"$|E| along x @ z = 0$")
ax.set_xlabel(r"$x (m)$")
ax.set_ylabel(r"$E_x (V/m)$")

# line2, = ax.plot(info.x, Ex_diag, label='$E_x$')
# line3, = ax.plot(info.x, Ey_diag, label='$E_y$')
# line4, = ax.plot(info.x, Ez_diag, label='$E_z$')
# lines.append(line2)
# lines.append(line3)
# lines.append(line4)

# ax.legend()

add_external_parameter_box(fig, static_params)

plt.show()

def update(k):
    lines[0].set_ydata(lineouts[k][1])
    it = iterations[k]
    t_us = series_f.t[k] * 1e6
    title.set_text(r'$E_x$, ' + f'@ y = z = 0, it {it:.3f} t = {t_us:.2f} us')

ani = animation.FuncAnimation(fig, update, frames=len(iterations), interval=100)
ani.save(f'{save_path}/Ex_lineout_timelapse.mp4', writer='ffmpeg', fps=10, dpi=300)
print(f"Saved: {save_path}/Ex_lineout_timelapse.mp4")

# $E_y$ on xz-plane

In [ ]:
eys = []

for it in iterations:
    print(f"It{it}", end='\r')
    Ex_diag, info = series_f.get_field(field='E', coord='x',slice_across=['y'], iteration=it)
    Ey_diag, _    = series_f.get_field(field='E', coord='y',slice_across=['y'], iteration=it)
    Ez_diag, _    = series_f.get_field(field='E', coord='z',slice_across=['y'], iteration=it)

    eys.append(Ey_diag)

xs = info.x
zs = info.z

vmin = -np.max(np.abs(eys))
vmax = -vmin

fig, (ax, ax_text) = plt.subplots(1, 2, figsize=(10, 5), width_ratios=(3, 1))

ax_text.axis('off')

lines = []

im = ax.imshow(eys[0], origin='lower',
               extent=[xs[0], xs[-1], zs[0], zs[-1]],
               vmin=vmin, vmax=vmax, cmap='RdBu_r', aspect='equal')
title = ax.set_title(r"$E_y$ on xz-plane")
ax.set_xlabel(r"$x (m)$")
ax.set_ylabel(r"$z (m)$")

fig.colorbar(im, label='$E_y$ (V/m)')

add_external_parameter_box(fig, static_params, dx = 6.5)

plt.show()

def update(k):
    im.set_data(eys[k])
    it = iterations[k]
    t_us = series_f.t[k] * 1e6
    title.set_text(r'$E_y$' + f', it {it:.3f} t = {t_us:.2f} us')

ani = animation.FuncAnimation(fig, update, frames=len(iterations), interval=100)
ani.save(f'{save_path}/Ey_xz_timelapse.mp4', writer='ffmpeg', fps=10, dpi=300)
print(f"Saved: {save_path}/Ey_xz_timelapse.mp4")

# $E_z$ on xy plane

In [ ]:
ezs = []

for it in iterations:
    print(f"It{it}", end='\r')
    Ex_diag, info = series_f.get_field(field='E', coord='x',slice_across=['z'], iteration=it)
    Ey_diag, _    = series_f.get_field(field='E', coord='y',slice_across=['z'], iteration=it)
    Ez_diag, _    = series_f.get_field(field='E', coord='z',slice_across=['z'], iteration=it)

    ezs.append(Ez_diag)

xs = info.x
ys = info.y

vmin = -np.max(np.abs(ezs))
vmax = -vmin

fig, (ax, ax_text) = plt.subplots(1, 2, figsize=(10, 5), width_ratios=(3, 1))

ax_text.axis('off')

lines = []

im = ax.imshow(ezs[0], origin='lower',
               extent=[xs[0], xs[-1], ys[0], ys[-1]],
               vmin=vmin, vmax=vmax, cmap='RdBu_r', aspect='equal')
title = ax.set_title(r"$E_z$ on xy-plane")
ax.set_xlabel(r"$x (m)$")
ax.set_ylabel(r"$y (m)$")

fig.colorbar(im, label='$E_z$ (V/m)')

add_external_parameter_box(fig, static_params, dx = 6.5)

plt.show()

def update(k):
    im.set_data(ezs[k])
    it = iterations[k]
    t_us = series_f.t[k] * 1e6
    title.set_text(r'$E_z$' + f', it {it:.3f} t = {t_us:.2f} us')

ani = animation.FuncAnimation(fig, update, frames=len(iterations), interval=100)
ani.save(f'{save_path}/Ez_xy_timelapse.mp4', writer='ffmpeg', fps=10, dpi=300)
print(f"Saved: {save_path}/Ez_xy_timelapse.mp4")

# $E_x$ on yz-plane

In [ ]:
exs = []

for it in iterations:
    print(f"It{it}", end='\r')
    Ex_diag, info = series_f.get_field(field='E', coord='x',slice_across=['x'], iteration=it)
    Ey_diag, _    = series_f.get_field(field='E', coord='y',slice_across=['x'], iteration=it)
    Ez_diag, _    = series_f.get_field(field='E', coord='z',slice_across=['x'], iteration=it)

    exs.append(Ex_diag)

ys = info.y
zs = info.z

vmin = -np.max(np.abs(exs))
vmax = -vmin

fig, (ax, ax_text) = plt.subplots(1, 2, figsize=(10, 5), width_ratios=(3, 1))

ax_text.axis('off')

lines = []

im = ax.imshow(exs[0], origin='lower',
               extent=[xs[0], xs[-1], ys[0], ys[-1]],
               vmin=vmin, vmax=vmax, cmap='RdBu_r', aspect='equal')
title = ax.set_title(r"$E_x$ on yz-plane")
ax.set_xlabel(r"$y (m)$")
ax.set_ylabel(r"$z (m)$")

fig.colorbar(im, label='$E_x$ (V/m)')

add_external_parameter_box(fig, static_params, dx = 6.5)

plt.show()

def update(k):
    im.set_data(exs[k])
    it = iterations[k]
    t_us = series_f.t[k] * 1e6
    title.set_text(r'$E_x$' + f', it {it:.3f} t = {t_us:.2f} us')

ani = animation.FuncAnimation(fig, update, frames=len(iterations), interval=100)
ani.save(f'{save_path}/Ex_yz_timelapse.mp4', writer='ffmpeg', fps=10, dpi=300)
print(f"Saved: {save_path}/Ex_yz_timelapse.mp4")

# $E_\perp$

In [ ]:
e_normals = []

for it in iterations:
    print(f"It{it}", end='\r')

    # x on yz-plane
    Ex_diag, yz_info    = series_f.get_field(field='E', coord='x',slice_across=['x'], iteration=it)

    # y on xz-plane
    Ey_diag, xz_info    = series_f.get_field(field='E', coord='y',slice_across=['y'], iteration=it)

    # z on xy-plane
    Ez_diag, xy_info    = series_f.get_field(field='E', coord='z',slice_across=['z'], iteration=it)

    e_normals.append([Ex_diag, Ey_diag, Ez_diag])

fig, (ax_yz, ax_xz, ax_xy, ax_text) = plt.subplots(1, 4, figsize=(20, 5), width_ratios=(2, 2, 2, 1), layout='constrained')
ax_text.axis('off')

# for info, field_data in zip([yz_info, xz_info, xy_info], [Ex_diag, Ey_diag, Ez_diag]):
#     print(info.axes)
#     for i, axis_name in enumerate(info.axes):
#         # info.dimensions gives the grid size along each axis
#         print(f"Axis {i} represents '{axis_name}' with size {field_data.shape[i]}")

xs = xz_info.x
ys = yz_info.y
zs = yz_info.z

# plot x on yz
vmin = -np.max(np.abs([e[0] for e in e_normals]))
vmax = -vmin

im_yz = ax_yz.imshow(e_normals[-1][0], origin='lower',
            extent=[ys[0], ys[-1], zs[0], zs[-1]],
            vmin=vmin, vmax=vmax, cmap='RdBu_r', aspect='equal')
title_yz = ax_yz.set_title(r"$E_x$ on yz-plane")
ax_yz.set_xlabel(r"$y (m)$")
ax_yz.set_ylabel(r"$z (m)$")

fig.colorbar(im_yz, label='$E_x$ (V/m)')

# plot y on xz-plane

vmin = -np.max(np.abs([e[1] for e in e_normals]))
vmax = -vmin

im_xz = ax_xz.imshow(e_normals[-1][1], origin='lower',
            extent=[xs[0], xs[-1], zs[0], zs[-1]],
            vmin=vmin, vmax=vmax, cmap='RdBu_r', aspect='equal')
title_xz = ax_xz.set_title(r"$E_y$ on xz-plane")
ax_xz.set_xlabel(r"$x (m)$")
ax_xz.set_ylabel(r"$z (m)$")

fig.colorbar(im_xz, label='$E_y$ (V/m)')

# plot z on xy-plane

vmin = -np.max(np.abs([e[2] for e in e_normals]))
vmax = -vmin

im_xy = ax_xy.imshow(e_normals[-1][2], origin='lower',
            extent=[xs[0], xs[-1], ys[0], ys[-1]],
            vmin=vmin, vmax=vmax, cmap='RdBu_r', aspect='equal')
title_xy = ax_xy.set_title(r"$E_z$ on xy-plane")
ax_xy.set_xlabel(r"$x (m)$")
ax_xy.set_ylabel(r"$y (m)$")

fig.colorbar(im_xy, label='$E_z$ (V/m)')

add_external_parameter_box(fig, params=static_params, dx = 6)

title = fig.suptitle(r'$E_\perp$')

plt.savefig('test.png')

plt.show()

def update(k):
    print(f"It{k/len(iterations):.2%}", end='\r')
    im_yz.set_data(e_normals[k][0])
    im_xz.set_data(e_normals[k][1])
    im_xy.set_data(e_normals[k][2])
    it = iterations[k]
    t_us = series_f.t[k] * 1e6
    title.set_text(r'$E_\perp$' + f', it {it:.3f} t = {t_us:.2f} us')

def animate(frame):
    pass

# 2. FORCE an initial canvas draw so constrained_layout calculates everything
fig.canvas.draw()

# 3. FREEZE the layout engine so it stops shifting frames during the video save
fig.set_layout_engine('none')

ani = animation.FuncAnimation(fig, update, frames=len(iterations), interval=100)
ani.save(f'{save_path}/E_normals_timelapse.mp4', writer='ffmpeg', fps=10, dpi=300)
print(f"Saved: {save_path}/E_normals_timelapse.mp4")